# NanoGPT (Learn)

In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [2]:
import os
import sys
from pathlib import Path

from pathlib import Path

CWD = os.path.realpath(os.getcwd())
PARENT_DIR = os.path.dirname(CWD)
sys.path.append(PARENT_DIR)

DATA_DIR = Path(PARENT_DIR).parent / 'data'

In [3]:
from reader.loader import TextDataset

dataset = TextDataset(DATA_DIR, device=device)

100%|██████████| 5/5 [00:00<00:00, 113.73it/s]


In [4]:
from src.modules.architecture.ngram_lm import NgramLanguageModel
from reader.preprocess import decode
import torch

BATCH_SIZE = 16
EMBEDDING_SIZE = 64
SEQ_LENGTH = 32
DROPOUT_RATE = 0.2

N_HEADS = 4
N_BLOCKS = 4
LR = 1e-3

EVAL_ITER = 100
EVAL_INTERVAL = 100
EPOCH_SIZE = 10000

torch.manual_seed(1337)

model = NgramLanguageModel(vocab_size=dataset.vocab_size, n_heads=N_HEADS, embedding_size=EMBEDDING_SIZE, seq_length=SEQ_LENGTH, 
                            n_blocks=N_BLOCKS, dropout_rate=DROPOUT_RATE, device=device, attention_type='causal', multi_attention_type='multiqueries')
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

print("Number of parameters:")
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

Number of parameters:
0.192345 M parameters


In [5]:
@torch.no_grad()
def estimate_loss(x, y, model):
    losses = torch.zeros(EVAL_ITER)
    for k in range(EVAL_ITER):
        logits, loss = model(x, y)
        losses[k] = loss.item()
    return losses.mean()

In [6]:
for iter in range(EPOCH_SIZE):

    # every once in a while evaluate the loss on train and val sets
    if iter % EVAL_INTERVAL == 0 or iter == EPOCH_SIZE - 1:
        model.eval()
        x_train, y_train = dataset.load_train(batch_size=BATCH_SIZE, context_window_size=SEQ_LENGTH)
        x_val, y_val = dataset.load_test(batch_size=BATCH_SIZE, context_window_size=SEQ_LENGTH)
        train_losses = estimate_loss(x_train, y_train, model)
        val_losses = estimate_loss(x_val, y_val, model)
        print(f"step {iter}: train loss {train_losses:.4f}, val loss {val_losses:.4f}")
        model.train()

    # sample a batch of data
    x_train, y_train = dataset.load_train(BATCH_SIZE)

    # evaluate the loss
    logits, loss = model(x_train, y_train)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.5657, val loss 4.5798
step 100: train loss 2.5314, val loss 2.5536
step 200: train loss 2.4492, val loss 2.4284
step 300: train loss 2.2775, val loss 2.3210
step 400: train loss 2.1343, val loss 2.1995
step 500: train loss 2.0709, val loss 2.0576
step 600: train loss 1.9449, val loss 2.0928
step 700: train loss 1.9080, val loss 1.9702
step 800: train loss 1.9245, val loss 1.9021
step 900: train loss 1.8330, val loss 1.8786
step 1000: train loss 1.7798, val loss 1.8525
step 1100: train loss 1.7437, val loss 1.8580
step 1200: train loss 1.7239, val loss 1.8613
step 1300: train loss 1.6821, val loss 1.8640
step 1400: train loss 1.7428, val loss 1.8248
step 1500: train loss 1.6684, val loss 1.8790
step 1600: train loss 1.7040, val loss 1.7928
step 1700: train loss 1.6542, val loss 1.7674
step 1800: train loss 1.6337, val loss 1.7579
step 1900: train loss 1.6525, val loss 1.7711
step 2000: train loss 1.6039, val loss 1.7962
step 2100: train loss 1.6399, val loss 1.7278


In [7]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))



Awfitu story in that the mottage, and providaption of itas. On pennsidenctival of all his fours, but nextreasive of the far? But herem, and doubt,fam a ilyoshy would counces for a not worms't able peopent, for her for Fyodor Pavlovitch Karamamazantage  of  thing,  or  the  due  to  sacress  islarge, it  from  menaint   solew.   He juded from instrentained, which the times a fellow him. Burge  in  its,  has  insiparaturations that is that aitto he from of nurge. Prounceles landwhy  explaintion;  and  only some veriage  with wooks,  there  in  ahe  allaseed in  when  the  expose to  than  extronguing;  hos a should nexcessuing  before  diesceponding  time  infimis  he accasessed   amuaTed  the  sometimes  this  out  othing.   .   One  likept;  I  have  "stall's  a  good-plaping  unator  that  if  mee.)..     VoRY   fuldh  need,  ifoon  of dORCUOBIO:

ATESTERIO:
Will what hable incoment a huncunion febous my not face, and contain'd side in his fourth deased witch or the lock. Metraight,

In [8]:
from reader.preprocess import encode
context = torch.tensor([encode(dataset.stoi, 'fyodor')], dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))

fyodor Pavlovitch, thousand to him; have supputole of woes shereed him betrace out had exagge—with, shaled he hard and more more probancedies, howerself-old into marriage, ind pertation. Supid words) sined, I'll minindas sugglooin gotting you fel. Him, Ivan the cempt him hused haba prounce, besert, if oncently because for a chily of all Chapraction fount into for the dismerel's stase any extrustions, recked for over timed his sidismaggestely of Him. Make up to heave Riods hare him Fyodor Pavlovitch's arphy Getle socisative its of stan onlides; to the “resies? who was like his tearlased with maininity, for her, about begin am or hesete but lay hasting defanctly getting a person thying and with dowruck in consomin upon hights as that intelling with Towauchin heas blews shevelly they were he was vened that he had nothing forphosate to gon of her radvanour:
Pogoes and-who straights of my heire four, he had entent years and tymaning, and all that wapauche was a garal. At with aturha Mitya i

In [9]:
context = torch.tensor([encode(dataset.stoi, 'slab')], dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))

slaboon him three 1 Ivan ovever a  sallf a housesiffes.  He  garrood, and  unaggerson  in the worthy  the  like the suppose, his  like of obething  the  drulack  for  a  language-commeng  will (of their one there of motherely was so Alyosha, as I paraga,  thing makis, woulo, the with greed and grarless. That when I have know from a witting. There hou little is han not panobrity.

Seem I Mia awally,
UEELI:
Loo!
“rible to generaivate up in cell to one him afterwards to suted in besirat that time their Paplagrast of characes that KeramazIver.” So Ivanovna waitna.

Fyodonosha say Alyosha's will monastery?

Wonld had brease.
God or a Ivans. Boys) every were he offfairing, fighous, have before. Are
was beht entenced and drevoic part Care the time sported for a indistenteance way, as a Year unterutterriage ier, of his was and that josts, and, went sign the broth Muscerely was all soulife,  we  thave  which  a  one  gone  that, we.perhaps of the  insipebrients  of  soun  of this  indrust  of l

In [10]:
torch.randint(
            100 - 10,
            (16,)
        )

tensor([84, 83, 87, 21, 74, 81, 39, 77, 58, 46, 19, 50, 82, 49, 37,  2])

In [11]:
context = torch.tensor([encode(dataset.stoi, 'slab')], dtype=torch.long, device=device)
idx_cond = context[:, -32:]

In [12]:
idx_cond

tensor([[72, 65, 54, 55]], device='cuda:0')